# Simple Wikipedia Dataset
Parse, categorize, and upload the Simple Wikipedia XML dump.

In [ ]:
import re
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

import networkx as nx
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datasets import Dataset, DatasetDict
from IPython.display import display
from ollama import Client
from tqdm import tqdm


## Parse XML

In [ ]:
CATEGORY_RE = re.compile(r"\[\[Category:(.*?)\]\]")
NS = "{http://www.mediawiki.org/xml/export-0.11/}"
XML_PATH = "../../data/simplewiki-latest-pages-articles.xml"


def wikipedia_articles(path=XML_PATH):
    for _, elem in ET.iterparse(path, events=("end",)):
        if elem.tag != NS + "page":
            continue
        if elem.find(NS + "ns").text != "0":
            elem.clear()
            continue

        text_elem = elem.find(NS + "revision/" + NS + "text")
        text = text_elem.text if text_elem is not None else None
        yield {
            "title": elem.find(NS + "title").text,
            "text": text,
            "categories": CATEGORY_RE.findall(text) if text else [],
        }
        elem.clear()


dataset = Dataset.from_list(list(wikipedia_articles()))
print(f"{len(dataset):,} articles loaded")

## Clean raw wikitext

Keep raw article text in `text`, add a markdown-ish `clean_text`, and optionally drop low-quality pages.

In [ ]:
import hashlib

MIN_CLEAN_TEXT_CHARS = 120
DROP_REDIRECTS = True
DROP_DISAMBIGUATION = True
DROP_EXACT_DUPES = True
DROP_LIST_HEAVY = False  # set True to exclude list-only / calendar-like pages

DISAMBIGUATION_HINTS = (
    " may refer to:",
    " can refer to:",
    " might refer to:",
    " may also refer to:",
)

try:
    import mwparserfromhell
except ImportError:
    mwparserfromhell = None

try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None


def is_redirect(text: str | None) -> bool:
    return bool(text and text.lstrip().lower().startswith("#redirect"))


def is_disambiguation(title: str | None, text: str | None) -> bool:
    if not text:
        return False
    title = (title or "").lower().strip()
    body = text.lower()
    return title.endswith("(disambiguation)") or any(
        hint in body[:500] for hint in DISAMBIGUATION_HINTS
    )


def strip_wikitable_blocks(text: str, replacement: str = "\n[Table omitted]\n") -> str:
    out = []
    i = 0
    n = len(text)
    while i < n:
        if text.startswith("{|", i):
            depth = 1
            j = i + 2
            while j < n and depth > 0:
                if text.startswith("{|", j):
                    depth += 1
                    j += 2
                elif text.startswith("|}", j):
                    depth -= 1
                    j += 2
                else:
                    j += 1
            out.append(replacement)
            i = j
        else:
            out.append(text[i])
            i += 1
    return "".join(out)


def normalize_whitespace(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def convert_headings(text: str) -> str:
    def repl(match):
        level = min(len(match.group(1)), 6)
        title = match.group(2).strip()
        return f"\n{'#' * level} {title}\n"

    return re.sub(r"(?m)^(={2,6})\s*(.*?)\s*\1\s*$", repl, text)


def clean_wikilinks(text: str) -> str:
    text = re.sub(r"\[\[([^|\]]+)\|([^\]]+)\]\]", r"\2", text)
    text = re.sub(r"\[\[([^\]]+)\]\]", r"\1", text)
    return text


def remove_external_links_keep_label(text: str) -> str:
    text = re.sub(r"\[(https?://[^\s\]]+)\s+([^\]]+)\]", r"\2", text)
    text = re.sub(r"\[(https?://[^\]]+)\]", "", text)
    return text


def remove_templates(text: str) -> str:
    prev = None
    while prev != text:
        prev = text
        text = re.sub(r"\{\{[^{}]*\}\}", "", text)
    return text


def regex_clean_wikitext(text: str) -> str:
    text = strip_wikitable_blocks(text)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.S)
    text = re.sub(r"<ref[^>]*>.*?</ref>", " ", text, flags=re.I | re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    text = remove_templates(text)
    text = re.sub(r"\[\[(?:File|Image):[^\]]+\]\]", " ", text, flags=re.I)
    text = re.sub(r"(?im)^\s*\[\[Category:[^\]]+\]\]\s*$", "", text)
    text = remove_external_links_keep_label(text)
    text = clean_wikilinks(text)
    text = convert_headings(text)
    text = re.sub(r"(?m)^\*+\s*", "- ", text)
    text = re.sub(r"(?im)^\s*__[^_]+__\s*$", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    return normalize_whitespace(text)


def parser_clean_wikitext(text: str) -> str:
    code = mwparserfromhell.parse(strip_wikitable_blocks(text))

    for tag in code.filter_tags(recursive=True):
        tag_str = str(tag).lower()
        if "<ref" in tag_str:
            code.remove(tag)
        else:
            try:
                contents = tag.contents.strip_code() if hasattr(tag, "contents") else ""
            except Exception:
                contents = ""
            code.replace(tag, contents)

    for tpl in code.filter_templates(recursive=True):
        code.remove(tpl)

    for link in code.filter_wikilinks(recursive=True):
        title = str(link.title).strip().lower()
        if title.startswith(("file:", "image:", "category:")):
            code.remove(link)

    cleaned = code.strip_code(normalize=True, collapse=True)
    cleaned = convert_headings(cleaned)
    cleaned = remove_external_links_keep_label(cleaned)
    cleaned = clean_wikilinks(cleaned)
    cleaned = remove_templates(cleaned)
    cleaned = re.sub(r"(?m)^\*+\s*", "- ", cleaned)
    if BeautifulSoup is not None:
        cleaned = BeautifulSoup(cleaned, "html.parser").get_text("\n")
    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    return normalize_whitespace(cleaned)


def is_list_heavy(text: str, threshold: float = 0.6) -> bool:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if not lines:
        return True
    bullet_lines = sum(1 for ln in lines if ln.startswith("- "))
    return (bullet_lines / len(lines)) >= threshold


def normalized_hash(text: str | None) -> str:
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r"\[\[category:.*?\]\]", " ", text, flags=re.I)
    text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def enrich_article(ex):
    if is_redirect(ex["text"]):
        return {
            "clean_text": "",
            "clean_char_count": 0,
            "is_redirect": True,
            "is_disambiguation": False,
            "is_list_heavy": False,
            "text_hash": normalized_hash(ex["text"]),
        }

    clean_text = regex_clean_wikitext(ex["text"])
    return {
        "clean_text": clean_text,
        "clean_char_count": len(clean_text),
        "is_redirect": is_redirect(ex["text"]),
        "is_disambiguation": is_disambiguation(ex["title"], ex["text"]),
        "is_list_heavy": is_list_heavy(clean_text),
        "text_hash": normalized_hash(ex["text"]),
    }


dataset = dataset.map(enrich_article, desc="Cleaning wikitext", num_proc=4)

if DROP_REDIRECTS:
    dataset = dataset.filter(
        lambda ex: not ex["is_redirect"], desc="Dropping redirects", num_proc=4
    )
if DROP_DISAMBIGUATION:
    dataset = dataset.filter(
        lambda ex: not ex["is_disambiguation"],
        desc="Dropping disambiguation pages",
        num_proc=4,
    )
dataset = dataset.filter(
    lambda ex: ex["clean_char_count"] >= MIN_CLEAN_TEXT_CHARS,
    desc="Dropping short cleaned pages",
)
if DROP_LIST_HEAVY:
    dataset = dataset.filter(
        lambda ex: not ex["is_list_heavy"], desc="Dropping list-heavy pages", num_proc=4
    )

if DROP_EXACT_DUPES:
    seen_hashes = set()

    def keep_first_duplicate(ex):
        text_hash = ex["text_hash"]
        if text_hash in seen_hashes:
            return False
        seen_hashes.add(text_hash)
        return True

    dataset = dataset.filter(
        keep_first_duplicate, desc="Dropping exact duplicates", num_proc=4
    )

print(f"{len(dataset):,} articles after cleanup")

## Text Length Distribution

In [ ]:
lengths = np.array([len(t) if t else 0 for t in dataset["clean_text"]])

PERCENTILES = [90, 95, 99, 99.9, 99.99]
pct_values = np.percentile(lengths, PERCENTILES)

print("Clean text length percentiles:")
for p, v in zip(PERCENTILES, pct_values):
    print(f"  p{p:<6} {v:>10,.0f} chars")

print(f"\n  min    {lengths.min():>10,} chars")
print(f"  median {np.median(lengths):>10,.0f} chars")
print(f"  mean   {lengths.mean():>10,.0f} chars")
print(f"  max    {lengths.max():>10,} chars")

# Top 5 largest articles
top5_idx = np.argsort(lengths)[-5:][::-1]
print("\nTop 5 longest articles:")
for rank, idx in enumerate(top5_idx, 1):
    print(f"  {rank}. {dataset[int(idx)]['title']!r:<50} {lengths[idx]:>10,} chars")

In [ ]:
fig1 = px.histogram(x=lengths, nbins=200, title="Full distribution")
fig1.update_yaxes(type="log", title="articles (log scale)")
for p, v in zip(PERCENTILES, pct_values):
    fig1.add_vline(
        x=v,
        line_dash="dash",
        annotation_text=f"p{p} = {v:,.0f}",
        annotation_position="top right",
    )

p99 = pct_values[PERCENTILES.index(99)]

fig2 = px.histogram(x=lengths[lengths <= p99], nbins=100, title="Below p99")
for p, v in zip(PERCENTILES, pct_values):
    fig2.add_vline(
        x=v,
        line_dash="dash",
        annotation_text=f"p{p} = {v:,.0f}",
        annotation_position="top right",
    )

fig3 = px.histogram(
    x=lengths[lengths > p99],
    nbins=100,
    title="Tail (above p99)",
    color_discrete_sequence=["salmon"],
)
for p, v in zip(PERCENTILES, pct_values):
    if v > p99:
        fig3.add_vline(
            x=v,
            line_dash="dash",
            annotation_text=f"p{p} = {v:,.0f}",
            annotation_position="top right",
        )

fig1.show()
fig2.show()
fig3.show()

## Tokenize data

Tokenize `clean_text` for modeling while preserving raw `text` for debugging and link extraction.

In [ ]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")

# dataset = dataset.map(
#     lambda x: {"tokens": tokenizer(x["clean_text"])["input_ids"]}, num_proc=2
# )

In [ ]:
from vllm import LLM, SamplingParams

# Load LLM model
model = LLM(
    "ibm-granite/granite-4.0-h-tiny",
    gpu_memory_utilization=0.8,
    quantization="fp8",
)

In [ ]:
system_prompt = """You are an expert technical writer specializing in creating educational content.

**CORE TASK**:
Convert a SimpleWikipedia passage into a comprehensive single-concept textbook chapter.

**TARGET AUDIENCE**: High school students with minimal prerequisite knowledge

**QUALITY CRITERIA**:
Exhibit these characteristics:
- **Clarity**: Each paragraph explains one idea (8th-grade reading level)
- **Scaffolding**: Concepts build from basic → intermediate  
- **Concrete**: Every abstract idea includes real-world example or analogy
- **Verifiable**: Claims are factually grounded in source material
- **Self-contained**: Minimal prerequisite knowledge needed

**OUTPUT STRUCTURE**:

## Title
[Concise, specific title]

## Learning Objectives
- [Specific skill/knowledge student will gain] (3 objectives)

## 1. Introduction (150-200 words)
Why care? When encountered? Include one motivating real-world context.

## 2. Foundational Concepts (300-400 words)
Define core terms: Definition → Example → Why it matters
Use analogies to familiar concepts.

## 3. Deep Dive: [Specific Subtopic] (400-600 words)
Expand on most important aspect.
Include step-by-step explanations, visual descriptions (text), edge cases.

## 4. Practical Applications (200-300 words)
2-3 realistic concrete examples.
Format: "[Scenario]: [How concept applies]"

## 5. Common Misconceptions (150-250 words)
3-4 things students often misunderstand.
Format: "**Misconception**: [belief] | **Reality**: [correct understanding]"

## 6. Key Takeaways
- Bullet points (5-8 max) addressing learning objectives

**TARGET LENGTH**: 1500-2000 words

**DIVERSITY CONSTRAINTS** (Critical for training data quality):
- Focus on ONE primary concept or application per chapter (not scattered subtopics)
- Vary section depth: if covering [Topic A], emphasize practical applications; if [Topic B], emphasize conceptual foundations
- Tailor "Practical Applications" to the specific domain hinted in the passage
- Avoid generic structures: customize section titles and emphasis based on content type

**WRITING GUIDELINES**:
- Active voice, direct address ("you") where appropriate
- Plain English; define technical terms on first use
- Numbered lists for sequences, bullets for collections
- Transitional sentences between sections
- Information grounded in passage and general educational knowledge only
"""

# Prepare batch of prompts (sample first 5000 articles to limit computation)
texts = dataset["clean_text"]
prompts = [
    [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"Generate a textbook for the following passage:\n\n{text}",
        },
    ]
    for text in texts
]

print(f"Generating textbooks for {len(prompts)} articles...")

# Generate textbooks in batches
responses = model.chat(
    prompts, sampling_params=SamplingParams(temperature=0, max_tokens=8192)
)

textbooks = [response.outputs[0].text for response in responses]
print(f"Generated {len(textbooks)} textbooks")


In [ ]:
# Tokenize the generated textbooks
print("Tokenizing textbooks...")

# Create a dataset slice with the generated textbooks
textbook_dataset = dataset.select(range(len(textbooks)))
textbook_dataset = textbook_dataset.add_column("textbook", textbooks)

# Tokenize textbooks
textbook_dataset = textbook_dataset.map(
    lambda x: {"textbook_tokens": tokenizer(x["textbook"])["input_ids"]}, num_proc=2
)

print(f"Tokenized textbooks: {len(textbook_dataset)} articles")
print(f"Sample textbook tokens: {len(textbook_dataset[0]['textbook_tokens'])} tokens")


## Tokenize Textbooks


## Generate Textbooks from Clean Text

Use LLM to expand cleaned articles into comprehensive textbooks.


## Upload to Hugging Face

In [ ]:
dataset_for_upload = textbook_dataset.shuffle(seed=42)

test_dataset = dataset_for_upload.select(range(min(1000, len(dataset_for_upload) // 5)))
val_dataset = dataset_for_upload.select(
    range(
        test_dataset.num_rows,
        test_dataset.num_rows + min(1000, len(dataset_for_upload) // 5),
    )
)
train_dataset = dataset_for_upload.select(
    range(val_dataset.data.index[-1] + 1, len(dataset_for_upload))
)

splits = DatasetDict({"train": train_dataset, "val": val_dataset, "test": test_dataset})

print(f"Uploading dataset splits:")
print(f"  Train: {len(train_dataset)} articles")
print(f"  Val: {len(val_dataset)} articles")
print(f"  Test: {len(test_dataset)} articles")

splits.push_to_hub("karanravindra/simple-wikipedia-textbooks")


## Summary

## Graph

In [ ]:
LINK_RE = re.compile(r"\[\[([^|\]#\n]+?)(?:\|[^\]]*?)?\]\]")
SKIP = (
    "Category:",
    "File:",
    "Image:",
    "Template:",
    "Wikipedia:",
    "Help:",
    "Talk:",
    "User:",
    "Special:",
    "Portal:",
)


def extract_links(text):
    if not text:
        return []
    out = []
    for m in LINK_RE.finditer(text):
        t = m.group(1).strip()
        if t and not any(t.startswith(p) for p in SKIP):
            out.append(t[0].upper() + t[1:])
    return out


title_set = set(dataset["title"])
in_degree = Counter()
link_counts = defaultdict(Counter)  # source → {target: count}

for row in tqdm(dataset, desc="Extracting links"):
    title = row["title"]
    for link in extract_links(row["text"]):
        if link in title_set and link != title:
            link_counts[title][link] += 1
            in_degree[link] += 1

print(f"Articles with outgoing links: {len(link_counts):,}")
print("Top 10 most-linked pages:")
for title, n in in_degree.most_common(10):
    print(f"  {n:>5,}  {title}")

In [ ]:
TOP_N = 400
top_nodes = {t for t, _ in in_degree.most_common(TOP_N)}

G = nx.DiGraph()
for src, targets in link_counts.items():
    if src not in top_nodes:
        continue
    for tgt, cnt in targets.items():
        if tgt in top_nodes:
            G.add_edge(src, tgt, weight=cnt)
for n in top_nodes:
    G.add_node(n)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

pos = nx.spring_layout(G, k=1.5 / G.number_of_nodes() ** 0.5, seed=42, iterations=60)

node_list = list(G.nodes())
node_indeg = [G.in_degree(n) for n in node_list]
node_outdeg = [G.out_degree(n) for n in node_list]

# Single trace for all edges (dim base layer)
ex, ey = [], []
for src, tgt in G.edges():
    x0, y0 = pos[src]
    x1, y1 = pos[tgt]
    ex += [x0, x1, None]
    ey += [y0, y1, None]

base_edges = go.Scatter(
    x=ex,
    y=ey,
    mode="lines",
    line=dict(width=0.6, color="rgba(150,150,150,0.2)"),
    hoverinfo="none",
)

# Empty trace — filled on click with top-5 connection edges
highlight_edges = go.Scatter(
    x=[],
    y=[],
    mode="lines",
    line=dict(width=2.5, color="rgba(255,140,0,0.9)"),
    hoverinfo="none",
)

node_trace = go.Scatter(
    x=[pos[n][0] for n in node_list],
    y=[pos[n][1] for n in node_list],
    mode="markers+text",
    text=node_list,
    textposition="top center",
    textfont=dict(size=8),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "In-links (received): %{customdata[0]}<br>"
        "Out-links (sent): %{customdata[1]}"
        "<extra></extra>"
    ),
    customdata=list(zip(node_indeg, node_outdeg)),
    marker=dict(
        size=[2 + min(d / 20, 12) for d in node_indeg],
        color=node_outdeg,
        colorscale="RdYlBu",
        reversescale=True,
        showscale=True,
        colorbar=dict(title="Out-links<br>(links sent)", thickness=14),
        line=dict(width=0.5, color="white"),
        opacity=[1.0] * len(node_list),
    ),
)

fig = go.FigureWidget([base_edges, highlight_edges, node_trace])
fig.update_layout(
    title=(
        f"Simple Wikipedia Link Graph — top {TOP_N} most-linked pages<br>"
        "<sub>Node size = in-degree · Color = out-degree · "
        "Click a node to highlight its top 5 connections</sub>"
    ),
    showlegend=False,
    hovermode="closest",
    height=850,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    margin=dict(l=20, r=20, t=70, b=20),
)


def on_click(trace, points, state):
    if not points.point_inds:
        return
    clicked = node_list[points.point_inds[0]]

    # Sum edge weights across both in and out edges
    connections = {}
    for src, tgt, d in G.edges(data=True):
        w = d.get("weight", 1)
        if src == clicked:
            connections[tgt] = connections.get(tgt, 0) + w
        elif tgt == clicked:
            connections[src] = connections.get(src, 0) + w

    top5 = set(sorted(connections, key=connections.get, reverse=True)[:5])
    highlight = {clicked} | top5

    # Build highlighted edge coordinates
    hx, hy = [], []
    for src, tgt in G.edges():
        if (src == clicked and tgt in top5) or (tgt == clicked and src in top5):
            x0, y0 = pos[src]
            x1, y1 = pos[tgt]
            hx += [x0, x1, None]
            hy += [y0, y1, None]

    with fig.batch_update():
        fig.data[2].marker.opacity = [
            1.0 if n in highlight else 0.08 for n in node_list
        ]
        fig.data[1].x = hx
        fig.data[1].y = hy


fig.data[2].on_click(on_click)
display(fig)

In [ ]:
client = Client()


def summarize(text: str, related: list[str] | None = None, wrap: bool = True) -> str:
    relation_ctx = ""
    if related:
        relation_ctx = (
            "\n\nThis article links to the following related pages: "
            + ", ".join(related[:10])  # cap to avoid prompt bloat
            + ".\nAfter summarizing the article, add 1 sentence describing "
            "how it connects to those related pages."
        )
    response = client.chat(
        model="granite4:tiny-h",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant that summarizes text "
                "and explains document relationships.",
            },
            {
                "role": "user",
                "content": (
                    f"Summarize the following article in 1-2 sentences."
                    f"{relation_ctx}\n\n{text}"
                ),
            },
        ],
    )
    summary = response.message.content.strip()
    if wrap:
        summary = "\n".join(re.findall(".{1,80}(?:\\s|$)", summary))
    return summary


# Demo: first article with its outgoing links as relational context
ex = dataset[0]
ex_links = list(link_counts.get(ex["title"], {}).keys())[:10]
print("\n".join(re.findall(".{1,80}(?:\\s|$)", ex["clean_text"][:500])))
print("-" * 80)
print(f"Links: {ex_links}")
print("-" * 80)
print(summarize(ex["clean_text"], related=ex_links))

In [ ]:
# output 20 cleaned text examples
for i in range(20):
    ex = splits["train"][i]
    print(
        f"{i + 1:>2}. {ex['title']!r} | raw={len(ex['text']):,} chars | clean={len(ex['clean_text']):,} chars"
    )
    print("-" * 80)
    print(ex["clean_text"][:3000])
    print("-" * 80)